<a href="https://colab.research.google.com/github/kushim2005/omniproject/blob/main/Image_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf Pillow
import fitz          # PyMuPDF
import json
import os
from PIL import Image
import io
from datetime import datetime,timezone

# ─── CONFIG ───────────────────────────────────────────────────────────
PDF_PATH   = "/content/Evaluating_Machine_Learning_Algorithms_to_Detect_and_Classify_DDoS_Attacks_in_IoT.pdf"          # <-- change to your PDF path
OUTPUT_DIR = "output/images"       # folder to save extracted images
META_FILE  = "output/metadata.json"

# ─── SETUP ────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("output", exist_ok=True)


def extract_images_from_pdf(pdf_path: str) -> list:
    """
    Extract all images from a PDF using PyMuPDF.
    Returns a list of metadata dictionaries.
    """
    doc = fitz.open(pdf_path)
    all_metadata = []
    image_count  = 0

    print(f"[INFO] Opened: {pdf_path}  ({len(doc)} pages)")

    for page_num in range(len(doc)):
        page     = doc[page_num]
        img_list = page.get_images(full=True)

        for img_index, img_info in enumerate(img_list):
            xref = img_info[0]

            # Extract raw image bytes
            base_image = doc.extract_image(xref)
            img_bytes  = base_image["image"]
            ext        = base_image["ext"]          # e.g. "png", "jpeg"

            # Open with Pillow for processing
            pil_img = Image.open(io.BytesIO(img_bytes))

            # Convert unusual modes to RGB for saving
            if pil_img.mode == "CMYK":
                pil_img = pil_img.convert("RGB")
            elif pil_img.mode == "P":
                pil_img = pil_img.convert("RGBA")

            # Skip tiny/blank images (noise)
            if pil_img.width < 50 or pil_img.height < 50:
                print(f"  [SKIP] Page {page_num+1}, img {img_index+1} - too small")
                continue

            # Build filename & save
            image_count += 1
            filename  = f"img_p{page_num+1:03d}_{img_index+1:02d}.png"
            save_path = os.path.join(OUTPUT_DIR, filename)
            pil_img.save(save_path, format="PNG")

            # Collect metadata
            metadata = {
                "image_id":        f"img_{image_count:04d}",
                "source_pdf":      pdf_path,
                "page_number":     page_num + 1,
                "image_index":     img_index + 1,
                "saved_path":      save_path,
                "original_format": ext,
                "width_px":        pil_img.width,
                "height_px":       pil_img.height,
                "color_mode":      pil_img.mode,
                "file_size_bytes": os.path.getsize(save_path),
                "extracted_at": datetime.now(timezone.utc).isoformat(),
            }
            all_metadata.append(metadata)

            print(f"  [SAVED] Page {page_num+1}, img {img_index+1} -> {filename}  "
                  f"({pil_img.width}x{pil_img.height})")

    doc.close()
    print(f"\n[DONE] Total images extracted: {image_count}")
    return all_metadata


def save_metadata(metadata_list: list, output_path: str):
    """Save image metadata to a JSON file."""
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(metadata_list, f, indent=2)
    print(f"[INFO] Metadata saved -> {output_path}")


# ─── MAIN ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Step 1: Extract images
    metadata = extract_images_from_pdf(PDF_PATH)

    # Step 2: Save metadata JSON
    save_metadata(metadata, META_FILE)

    # Step 3: Quick summary
    print("\n-- Summary ------------------------------------------")
    print(f"  Images extracted : {len(metadata)}")
    print(f"  Saved to         : {OUTPUT_DIR}/")
    print(f"  Metadata file    : {META_FILE}")
    print("-----------------------------------------------------")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 70.9 MB/s eta 0:00:00
[INFO] Opened: /content/Evaluating_Machine_Learning_Algorithms_to_Detect_and_Classify_DDoS_Attacks_in_IoT.pdf  (5 pages)
  [SAVED] Page 2, img 1 -> img_p002_01.png  (456x334)
  [SAVED] Page 3, img 1 -> img_p003_01.png  (1315x116)
  [SAVED] Page 4, img 1 -> img_p004_01.png  (535x353)
  [SAVED] Page 4, img 2 -> img_p004_02.png  (603x356)

[DONE] Total images extracted: 4
[INFO] Metadata saved -> output/metadata.json

-- Summary ------------------------------------------
  Images extracted : 4
  Saved to         : output/images/
  Metadata file    : output/metadata.json
-----------------------------------------------------
